# TrialMatch Colab runner
Launcher only: no logic here. Runtime > Change runtime type > T4 GPU. Run cells top to bottom; the last cells are separate steps (smoke test, train, evaluate).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = '/content/TrialMatch-LM'
if os.path.isdir(REPO + '/.git'):
    !git -C {REPO} pull
else:
    !git clone https://github.com/Taha-tech05/TrialMatch-LM.git {REPO}
%cd {REPO}

In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt
!pip install -q -e .

In [ ]:
import os
OUT = '/content/drive/MyDrive/trialmatch'   # checkpoints, logs, eval outputs persist here
os.environ['TRIALMATCH_OUT'] = OUT
os.makedirs(OUT, exist_ok=True)

In [ ]:
!python scripts/prepare_data.py   # rebuilds data/processed from the public dataset (processed files are not in git)

## 1. Smoke test (decides 4B vs 1.7B)

In [ ]:
!python scripts/smoke_test_train.py

## 2. Train (resumes automatically if the session was interrupted: just re-run this cell)
If the smoke test recommended the fallback, use `--model Qwen/Qwen3-1.7B`.

In [ ]:
!python scripts/train.py

## 3. Evaluate on the validation split (never the test split here)

In [ ]:
!python scripts/evaluate.py --tag base

In [ ]:
!python scripts/evaluate.py --adapter {OUT}/checkpoints/qwen3-4b_qlora_4bit/adapter_final --tag finetuned